# Groceries - Data Preparation and Transformation with Restrictions

This file is used to prepare and transform the groceries data. The following restrictions apply to this file:

## Restrictions:

1. Baskets with less than 30 items
2. Customers with less than 5 baskets

## Main Process and Steps:

### 1. Parameter Setup and Data Overview:
Set up the packages, path, and dataset name. It also includes an overview of the data, such as the date period of the data.

### 2. Indexing of Items:
Create indices to represent the items and a mapping table for reference.

### 3. Indexing of Customers:
Create indices to represent the customer numbers and a mapping table for reference.

### 4. Baskets for Each Customer and Items in Each Basket:
By utilizing timestamps, it is possible to determine which items were purchased together, group them into baskets, and identify the number of baskets each customer has.

### 5. Apply Restrictions:
Apply the specified restrictions to the data.

### 6. Separate Train and Test Datasets:
Split the dataset so that the number of baskets in the training set is equal to the number of baskets in the testing set. If a customer has an odd number of baskets, delete the latest date (last row).

### 7. Generate 3 Files as Input for the Model:
Generate the following files: `train_u2b.txt`, `train_b2i.txt`, and `test_b2i.txt`.

## Coding

### 1. Parameter Setup and Data Overview:

In [60]:
# Import packages
import pandas as pd
import numpy as np
import random

In [61]:
# Set path
# Set path
path_name = '../../DataPreparationandTransformation/groceries_data/data/grocerieswithres'

In [62]:
# Set and read dataset
df = pd.read_csv(path_name + '/Groceries_dataset.csv')
df.head(5)

,Member_number,Date,itemDescription
0,1808,21-07-2015,tropical fruit
1,2552,05-01-2015,whole milk
2,2300,19-09-2015,pip fruit
3,1187,12-12-2015,other vegetables
4,3037,01-02-2015,whole milk


In [63]:
# The date period of the data
# Convert 'Date' column to datetime format
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

# Find first and last dates of data
first_date = df['Date'].min()
last_date = df['Date'].max()

print(f"First date in the dataset: {first_date}")
print(f"Last date in the dataset: {last_date}")

First date in the dataset: 2014-01-01 00:00:00
Last date in the dataset: 2015-12-30 00:00:00


### 2. Indexing of Items:

In [64]:
# Factorize the itemDescription column
df['item_no'], item_labels = pd.factorize(df['itemDescription'])

# Create a mapping table
item_mapping = pd.DataFrame({
    'item_no': range(len(item_labels)),
    'itemDescription': item_labels
})

item_mapping.to_csv(path_name+r'/item_index_mapping.txt', sep='\t', index=False)
# Drop the original itemDescription column
df = df.drop(columns=['itemDescription'])
df

,Member_number,Date,item_no
0,1808,2015-07-21,0
1,2552,2015-01-05,1
2,2300,2015-09-19,2
3,1187,2015-12-12,3
4,3037,2015-02-01,1
...,...,...,...
38760,4471,2014-10-08,75
38761,2022,2014-02-23,64
38762,1097,2014-04-16,153
38763,1510,2014-12-03,11


### 3. Indexing of Customers:

In [65]:
# Factorize the Member_number column
df['uid'], item_labels = pd.factorize(df['Member_number'])

# Create a mapping table
item_mapping = pd.DataFrame({
    'Member_number': item_labels,
    'uid': range(len(item_labels))
})

# Drop the original Member_number column
df = df.drop(columns=['Member_number'])
df

,Date,item_no,uid
0,2015-07-21,0,0
1,2015-01-05,1,1
2,2015-09-19,2,2
3,2015-12-12,3,3
4,2015-02-01,1,4
...,...,...,...
38760,2014-10-08,75,1154
38761,2014-02-23,64,81
38762,2014-04-16,153,2768
38763,2014-12-03,11,301


### 4. Baskets for Each Customer and Items in Each Basket:

In [66]:
# Convert the 'Date' column to datetime format
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

# Group data by 'uid' and 'Date' to create baskets for each customer and items in each basket
grouped_df = df.groupby(['uid', 'Date'])['item_no'].apply(list).reset_index().sort_values(by=['uid', 'Date'])

expanded_df = grouped_df['item_no'].apply(pd.Series).rename(columns=lambda x: str(x+1))
expanded_df = expanded_df.fillna('')

for col in expanded_df.columns[:]:
    expanded_df[col] = expanded_df[col].apply(lambda x: str(int(x)) if x != '' else x)

final_df = pd.concat([grouped_df[['uid', 'Date']], expanded_df], axis=1) 

print("Columns in final_df:", final_df.columns)
print(final_df.head())

Columns in final_df: Index(['uid', 'Date', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11'], dtype='object')
   uid       Date    1    2    3 4 5 6 7 8 9 10 11
0    0 2014-11-29   56    1                       
1    0 2014-12-15   34    6  113                  
2    0 2015-02-04  118  102                       
3    0 2015-07-21    0    4   64                  
4    1 2014-02-19    3   13                       


### 5. Apply Restrictions:

In [67]:
# Determine the item columns, excluding 'uid' and 'Date'
item_columns = [col for col in final_df.columns if col not in ['uid', 'Date']]
print("Item columns:", item_columns)

# Calculate the count of valid items for each row
final_df['valid_count'] = final_df[item_columns].apply(
    lambda row: row.notna().sum(), axis=1
)

# Filter baskets where the number of valid items is <= 31
final_df = final_df.query('valid_count <= 31')
print("Columns in final_df after filtering:", final_df.columns)

# Function to trim item columns to a specified maximum count
def trim_columns(row, max_count):
    """
    Trim a row to include only valid non-empty values up to max_count.
    Fill the remaining columns with empty strings ('') if needed.
    """
    # Filter out empty or NaN values
    valid_values = [x for x in row if pd.notna(x) and x != '']
    # Trim to max_count and pad with empty strings
    trimmed = valid_values[:max_count]
    return trimmed + [''] * (max_count - len(trimmed))

# Determine the maximum count of valid items across all baskets
max_valid_count = final_df['valid_count'].max()
print("Max valid count:", max_valid_count)

# Apply trimming to item columns
trimmed_data = final_df[item_columns].apply(
    lambda row: trim_columns(row, max_valid_count), axis=1
)

# Create a new DataFrame for the trimmed item columns
df_trimmed = pd.DataFrame(
    trimmed_data.tolist(),
    index=final_df.index,
    columns=[f'item_{i+1}' for i in range(max_valid_count)]
)
print("Columns in df_trimmed:", df_trimmed.columns)

# Concatenate the key columns ('uid' and 'Date') with the trimmed item columns
final_df = pd.concat([final_df[['uid', 'Date']].reset_index(drop=True), df_trimmed], axis=1)
print("Columns in final_df after concat:", final_df.columns)

# Drop the auxiliary 'valid_count' column
final_df = final_df.drop(columns=['valid_count'], errors='ignore')

# Verify the resulting DataFrame
print(final_df.head())

Item columns: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11']
Columns in final_df after filtering: Index(['uid', 'Date', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11',
       'valid_count'],
      dtype='object')
Max valid count: 11
Columns in df_trimmed: Index(['item_1', 'item_2', 'item_3', 'item_4', 'item_5', 'item_6', 'item_7',
       'item_8', 'item_9', 'item_10', 'item_11'],
      dtype='object')
Columns in final_df after concat: Index(['uid', 'Date', 'item_1', 'item_2', 'item_3', 'item_4', 'item_5',
       'item_6', 'item_7', 'item_8', 'item_9', 'item_10', 'item_11'],
      dtype='object')
   uid       Date item_1 item_2 item_3 item_4 item_5 item_6 item_7 item_8  \
0    0 2014-11-29     56      1                                             
1    0 2014-12-15     34      6    113                                      
2    0 2015-02-04    118    102                                             
3    0 2015-07-21      0      4     64                               

In [68]:
# Customers with less than 5 baskets
# Keep the first 10 rows
final_df = final_df.groupby('uid').apply(lambda x: x.head(10)).reset_index(drop=True)
final_df

C:\Users\admin\AppData\Local\Temp\ipykernel_3556\3845527340.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_df = final_df.groupby('uid').apply(lambda x: x.head(10)).reset_index(drop=True)


,uid,Date,item_1,item_2,item_3,item_4,item_5,item_6,item_7,item_8,item_9,item_10,item_11
0,0,2014-11-29,56,1,,,,,,,,,
1,0,2014-12-15,34,6,113,,,,,,,,
2,0,2015-02-04,118,102,,,,,,,,,
3,0,2015-07-21,0,4,64,,,,,,,,
4,1,2014-02-19,3,13,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14953,3893,2014-08-19,64,37,,,,,,,,,
14954,3894,2014-10-20,37,17,,,,,,,,,
14955,3895,2014-02-22,75,69,,,,,,,,,
14956,3896,2014-05-23,71,141,,,,,,,,,


### 6. Separate Train and Test Datasets:

In [69]:
# ===== Step 1: Global Train/Test Split =====
train_rows = []
test_rows = []
global_basket_number = 0

# Go through each user
for uid, user_data in final_df.groupby('uid'):
    user_data_sorted = user_data  # assume already sorted by date

    if len(user_data_sorted) < 2:
        continue  # Skip users with less than 2 baskets

    # Split
    train_data = user_data_sorted.iloc[:-1].copy()
    test_data = user_data_sorted.iloc[-1:].copy()

    # Assign basket_number (global continuous index)
    train_data['basket_number'] = range(global_basket_number, global_basket_number + len(train_data))
    global_basket_number += len(train_data)

    test_data['basket_number'] = [global_basket_number]
    global_basket_number += 1

    train_rows.append(train_data)
    test_rows.append(test_data)

# Merge all users' train and test data
train_df = pd.concat(train_rows).reset_index(drop=True)
test_df = pd.concat(test_rows).reset_index(drop=True)


In [70]:
print("Training Data:")
print(train_df.head())

Training Data:
   uid       Date item_1 item_2 item_3 item_4 item_5 item_6 item_7 item_8  \
0    0 2014-11-29     56      1                                             
1    0 2014-12-15     34      6    113                                      
2    0 2015-02-04    118    102                                             
3    1 2014-02-19      3     13                                             
4    1 2014-06-20     83      1     95                                      

  item_9 item_10 item_11  basket_number  
0                                     0  
1                                     1  
2                                     2  
3                                     4  
4                                     5  


In [71]:
print("\nTesting Data:")
print(test_df.head())



Testing Data:
   uid       Date item_1 item_2 item_3 item_4 item_5 item_6 item_7 item_8  \
0    0 2015-07-21      0      4     64                                      
1    1 2015-01-05      1      0     13                                      
2    2 2015-09-19      2      3     33                                      
3    3 2015-12-12      3     65     70                                      
4    4 2015-02-23     84     28                                             

  item_9 item_10 item_11  basket_number  
0                                     3  
1                                     8  
2                                    12  
3                                    15  
4                                    17  


### 7. Generate 3 Files as Input for the Model:

In [72]:
# train_b2i
train_b2i = train_df.drop(columns=['uid'])

# Assign 'basket_number' as the first column
columns = ['basket_number'] + [col for col in train_b2i.columns if col != 'basket_number']
train_b2i = train_b2i[columns]

# Replace NaN with empty strings
train_b2i = train_b2i.fillna('')

# Convert all values to strings, handling empty values and non-string types
def clean_and_convert(value):
    if pd.isna(value) or value == '':
        return ''
    try:
        if isinstance(value, (str, int)):
            return str(int(value))
        elif isinstance(value, float) and not pd.isna(value):
            return str(int(value))
        else:
            return ''
    except ValueError:
        return ''

for col in train_b2i.columns[1:]:
    train_b2i[col] = train_b2i[col].apply(clean_and_convert)

# Formatting data
def format_row(row):
    return ' '.join(str(x) for x in row if x != '')

# Add timestamps
train_b2i['timestamp'] = train_df['Date'].dt.strftime('%Y%m%d')

formatted_rows = train_b2i.apply(lambda row: ' '.join(str(x).strip() for x in row if x != ''), axis=1)

# formatted_rows = train_b2i.apply(format_row, axis=1)
def validate_timestamp(row):
    try:

        return row
    except ValueError:
        return None

formatted_rows = formatted_rows.apply(validate_timestamp).dropna()  


# Export the data as a text file without column names
with open(f'{path_name}/train_b2i.txt', 'w') as file:
    for row in formatted_rows:
        file.write(row + '\n')

print(f'Data exported to {path_name}\\train_b2i.txt')

Data exported to ../../DataPreparationandTransformation/groceries_data/data/grocerieswithres\train_b2i.txt


In [73]:
# Test set - test_b2i
# Drop the original 'uid' column 
test_b2i = test_df.drop(columns=['uid'])

# Assign 'basket_number' as the first column
columns = ['basket_number'] + [col for col in test_b2i.columns if col != 'basket_number']
test_b2i = test_b2i[columns]

# Replace NaN with empty strings
test_b2i = test_b2i.fillna('')

# Convert all values to strings, handling empty values and non-string types
for col in test_b2i.columns[1:]:  
    test_b2i[col] = test_b2i[col].apply(clean_and_convert)

# Formatting function to join values with space, ignoring empty strings
def format_row(row):
    return ' '.join(str(x) for x in row if x != '')

# Add timestamps
test_b2i['timestamp'] = test_df['Date'].dt.strftime('%Y%m%d')

formatted_rows = test_b2i.apply(lambda row: ' '.join(str(x).strip() for x in row if x != ''), axis=1)
formatted_rows = formatted_rows.apply(validate_timestamp).dropna()  

# Formatting data
# formatted_rows = test_b2i.apply(format_row, axis=1)

# Export the data as a text file without column names
with open(f'{path_name}/test_b2i.txt', 'w') as file:
    for row in formatted_rows:
        file.write(row + '\n')

print(f'Data exported to {path_name}\\test_b2i.txt')

Data exported to ../../DataPreparationandTransformation/groceries_data/data/grocerieswithres\test_b2i.txt


In [74]:
# ===== Step 4: train_u2b =====

# Group and expand: uid -> basket list
grouped_df = train_df.groupby('uid')['basket_number'].apply(list).reset_index()

# Expand baskets
expanded_df = grouped_df['basket_number'].apply(pd.Series)
expanded_df.columns = [f'basket_{i+1}' for i in expanded_df.columns]

# Concatenate uid + baskets
train_u2b = pd.concat([grouped_df['uid'], expanded_df], axis=1)

# Replace NaN
train_u2b = train_u2b.fillna('')

# Clean and convert
for col in train_u2b.columns[1:]:  # Skip uid
    train_u2b[col] = train_u2b[col].apply(clean_and_convert)

# Formatting
formatted_rows = train_u2b.apply(format_row, axis=1)

# Export to txt
with open(f'{path_name}/train_u2b.txt', 'w') as file:
    for row in formatted_rows:
        file.write(row + '\n')

print(f'Data exported to {path_name}/train_u2b.txt')


Data exported to ../../DataPreparationandTransformation/groceries_data/data/grocerieswithres/train_u2b.txt


In [75]:
# ===== Step 5: test_u2b =====

# Only take ['uid', 'basket_number']
test_u2b = test_df[['uid', 'basket_number']].copy()

# Replace NaN
test_u2b = test_u2b.fillna('')

# Clean and convert
for col in test_u2b.columns[1:]:  # Skip uid
    test_u2b[col] = test_u2b[col].apply(clean_and_convert)

# Formatting
formatted_rows = test_u2b.apply(format_row, axis=1)

# Export to txt
with open(f'{path_name}/test_u2b.txt', 'w') as file:
    for row in formatted_rows:
        file.write(row + '\n')

print(f'Data exported to {path_name}/test_u2b.txt')



Data exported to ../../DataPreparationandTransformation/groceries_data/data/grocerieswithres/test_u2b.txt
